# 02.03 PyPTO 编程范式与 MPMD 执行模型

PyPTO 算子开发的关键不在于把普通 Python 语句逐行搬到设备上，而在于用 Python 函数描述 Tensor 计算，再由 PyPTO 记录、编译并调度执行。本节重点解释这种编程范式：哪些代码在 Host 侧运行，哪些代码进入 Kernel 描述，以及 MPMD 执行模型为什么适合组织不同类型的设备任务。

本节围绕三个问题展开：

1. Host 侧代码和 Kernel 侧代码如何分工。
2. JIT kernel 中的 Tensor 表达为什么不是普通 Python 立即执行语句。
3. MPMD 如何把不同类型的计算任务按依赖关系组织起来。


## 1. 最小运行时检查

本节主要讲解编程范式和执行模型，只需要确认 `torch` 与 `pypto` 可以导入。这里不重复完整环境初始化，也不设置具体 NPU 设备。


In [1]:
import torch
import pypto

print("torch:", torch.__version__)
print("pypto:", pypto.__file__)


torch: 2.7.1+cpu
pypto: /usr/local/lib/python3.12/dist-packages/pypto/__init__.py


/usr/local/lib/python3.12/dist-packages/torch_npu/__init__.py:324: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


输出只用于确认运行时依赖可用。后续内容重点在代码结构和执行模型，即使不执行 NPU kernel，也可以理解本节概念。


## 2. 从普通 Python 到 PyPTO JIT

普通 Python 函数通常按语句顺序立即执行。例如：

```python
z = x + y
```

如果 `x` 和 `y` 是普通 Python 对象，这行语句会立刻完成一次加法。PyPTO kernel 中的 Tensor 表达不同。被 `@pypto.frontend.jit` 修饰的函数会被 PyPTO 记录为计算描述，框架根据这些描述生成后续执行所需的设备任务。

可以先把 PyPTO JIT 理解为两个阶段：

| 阶段 | 作用 |
| --- | --- |
| 描述阶段 | 使用 Python 函数写出 Tensor 之间的计算关系 |
| 执行阶段 | PyPTO 将计算关系编译、调度并在 NPU 或 SIM 模式下运行 |

因此，JIT 函数中的代码既有 Python 的书写形式，又具有设备计算描述的含义。


## 3. Host 侧和 Kernel 侧

一个 PyPTO 示例通常包含 Host 侧代码和 Kernel 侧代码。Host 侧代码运行在普通 Python 环境中，负责准备数据、选择设备、调用 kernel 和验证结果。Kernel 侧代码由 `@pypto.frontend.jit` 标记，描述需要交给 PyPTO 编译和执行的 Tensor 运算。

| 类型 | 主要职责 | 典型语句 |
| --- | --- | --- |
| Host 侧 | 导入依赖、解析参数、创建 Tensor、调用 kernel、比较结果 | `torch.rand(...)`、`torch.empty(...)`、`assert_allclose(...)` |
| Kernel 侧 | 描述 Tensor 计算、设置执行组织、写回输出 | `pypto.set_vec_tile_shapes(...)`、`out[:] = x + y` |

这个分界是理解 PyPTO 程序结构的第一步。`torch.rand` 创建真实数据，属于 Host 侧；`out[:] = x + y` 描述设备侧计算，属于 Kernel 侧。Host 侧负责“把任务准备好”，Kernel 侧负责“说明任务怎么算”。


## 4. Kernel 函数签名的含义

Kernel 函数的参数通常写成：

```python
def add_kernel(
    x: pypto.Tensor([...], pypto.DT_FP32),
    y: pypto.Tensor([...], pypto.DT_FP32),
    out: pypto.Tensor([...], pypto.DT_FP32),
):
    ...
```

这里的 `pypto.Tensor([...], pypto.DT_FP32)` 不是创建 Tensor 数据，而是描述进入 kernel 的参数类型。真实 Tensor 数据已经在 Host 侧通过 PyTorch 创建，并在调用 kernel 时传入。

`[...]` 表示这里不在函数签名中固定具体 shape；`pypto.DT_FP32` 表示参数元素类型是 FP32。这个写法让 PyPTO 能够理解 kernel 的输入输出接口。


## 5. Hello World 中体现的最小 API 骨架

本节只关注 Hello World 为了表达编程范式必须出现的 API。更完整的 API 分类放在 `00.04` 中统一整理。

| API 或写法 | 在编程范式中的作用 |
| --- | --- |
| `@pypto.frontend.jit(...)` | 标记 Kernel 侧计算描述入口 |
| `pypto.RunMode.NPU` / `pypto.RunMode.SIM` | 指定 kernel 的运行后端 |
| `pypto.Tensor([...], pypto.DT_FP32)` | 描述 kernel 输入输出参数 |
| `pypto.set_vec_tile_shapes(...)` | 描述设备侧分块执行组织 |
| `out[:] = x + y` | 描述 Tensor 计算并写回输出 |

这些写法共同构成一个最小 kernel：JIT 入口说明“这段函数交给 PyPTO 处理”，参数标注说明“输入输出长什么样”，Tile 设置说明“计算如何组织”，输出写回说明“结果放到哪里”。


## 6. Tensor 表达在编程范式中的位置

加法 kernel 的核心表达式是：

```python
out[:] = x + y
```

在 Host 侧，真实输入 Tensor 已经准备好；在 Kernel 侧，这行代码描述逐元素加法和输出写回。这里不需要把它理解成普通 Python 对象之间的一次立即赋值，而应理解成设备侧计算描述的一部分。

本节只强调它在 Host/Kernel 分工中的位置：`x`、`y` 和 `out` 是 kernel 参数，`x + y` 是计算关系，`out[:] = ...` 是输出落点。关于 Operation 节点、计算图层次和图转换过程，下一节会集中展开。


## 7. Tile Shape 与执行组织

Kernel 中常见的 Tile 设置语句是：

```python
pypto.set_vec_tile_shapes(1, 4, 1, 64)
```

Tile Shape 可以理解为设备执行时组织数据的小块形状。它不改变 `out = x + y` 的数学含义，只影响底层如何分块执行。

对于 shape 为 `(1, 4, 1, 64)` 的逐元素加法，输入和输出共有 256 个元素。`set_vec_tile_shapes(1, 4, 1, 64)` 表示这个向量类计算可以按这组维度组织。后续遇到矩阵乘法时，会出现面向 Cube 类计算的 Tile 设置；逐元素计算和矩阵计算的执行组织方式不同，因此 Tile API 也不同。


## 8. MPMD 的直观含义

MPMD 是 Multiple Program Multiple Data 的缩写，可以理解为“多个程序片段处理多份数据”。它强调不同任务可以使用不同的程序片段，而不是所有计算单元都执行完全相同的代码。

| 模型 | 直观含义 |
| --- | --- |
| SPMD | 同一段程序在多个处理单元上处理不同数据 |
| MPMD | 不同程序片段按依赖关系处理不同数据 |

在 PyPTO 项目中，不同算子会形成不同类型的任务。逐元素加法主要是向量类计算；矩阵乘法更接近 Cube 类计算；Softmax 会组合最大值、减法、指数、求和和除法；复杂网络片段还可能同时包含矩阵计算、逐元素计算、规约和数据搬运。MPMD 模型的价值在于让这些不同任务按照依赖关系组合起来。

| 算子或计算片段 | 可能包含的任务类型 |
| --- | --- |
| Hello World 加法 | 逐元素加法 |
| MatMul + Bias | 矩阵乘法、逐元素加法 |
| Row Sum | 规约求和 |
| Softmax | 最大值规约、减法、指数、求和、除法 |
| Transformer Block 片段 | MatMul、归一化、激活、残差加法、数据重排 |

这些任务的执行方式并不相同。MPMD 模型使框架能够根据任务类型和依赖关系组织执行，而不是把所有设备核都强行套入同一段程序。


## 9. 从 Kernel 描述到 MPMD 执行

PyPTO 的执行可以概括为下面的流程：

```text
Kernel 中的 Tensor 表达
  -> 框架记录计算关系
  -> 生成设备侧任务
  -> 按依赖关系调度执行
```

在 Hello World 加法算子中，任务很小，主要就是一次逐元素加法。虽然示例简单，但它已经包含完整链路：Host 侧创建输入输出，Kernel 侧描述计算，PyPTO 记录计算关系，再把任务交给设备执行。

复杂算子的差别在于任务类型更多。例如一个矩阵乘法加激活函数的算子，可能同时包含矩阵计算、逐元素加法和逐元素激活。MPMD 的作用就是在这些不同任务之间建立有序执行关系。


## 10. 哪些语句属于 Kernel 描述

判断一段代码是否属于 PyPTO kernel 描述，可以先看它是否位于 `@pypto.frontend.jit` 修饰的函数内部，以及它是否描述 Tensor 计算或执行组织。

| 语句 | 所属位置 | 是否属于 Kernel 描述 | 原因 |
| --- | --- | --- | --- |
| `torch.rand(...)` | Host 侧 | 否 | 创建真实输入数据 |
| `torch.empty(...)` | Host 侧 | 否 | 分配输出缓冲区 |
| `pypto.set_vec_tile_shapes(...)` | Kernel 侧 | 是 | 描述设备侧执行组织 |
| `out[:] = x + y` | Kernel 侧 | 是 | 描述 Tensor 计算和输出写回 |
| `assert_allclose(...)` | Host 侧 | 否 | 比较结果是否正确 |

这个表格反映了 PyPTO 编程范式的核心：Host 侧负责准备和验证，Kernel 侧负责表达计算。


## 11. 编程范式小结

PyPTO 的基础编程范式可以概括为三句话：

1. Host 侧创建输入输出 Tensor，并负责调用和验证。
2. Kernel 侧用 JIT 函数描述 Tensor 计算。
3. 框架根据 Kernel 描述生成设备任务，并以 MPMD 方式组织不同类型的计算。

Hello World 加法算子的数学表达很简单，但它已经展示了 PyPTO 程序的基本结构。后续更复杂的算子仍然遵循同一条主线，只是 Kernel 中的计算关系更多，Tile 设置更丰富，MPMD 执行中需要组织的任务类型也更多。


## 12. 概念检查

| 代码或概念 | 分类 |
| --- | --- |
| `get_device_id()` | Host 侧辅助逻辑 |
| `torch.rand(shape, dtype=torch.float).to(device)` | Host 侧数据构造与搬运 |
| `@pypto.frontend.jit(...)` | Kernel 侧计算描述入口 |
| `pypto.Tensor([...], pypto.DT_FP32)` | Kernel 参数描述 |
| `pypto.set_vec_tile_shapes(1, 4, 1, 64)` | Kernel 执行组织描述 |
| `out[:] = x + y` | Kernel 计算表达和输出写回 |
| `golden = torch.add(...)` | Host 侧参考实现 |
| `assert_allclose(...)` | Host 侧验证逻辑 |

分类原则是：创建真实数据、打印信息、解析参数或比较结果，通常属于 Host 侧；位于 JIT 函数内部，并描述 Tensor 运算或执行组织的语句，通常属于 Kernel 侧。
